# **🍿 IMDb Top 10,000 Movies & TV Series**
This dataset contains structural metadata for the top 9,999 highest-rated and most-voted titles on IMDb, spanning movies, TV series, and TV movies. It is designed specifically for data scientists, analysts, and machine learning engineers looking to explore trends in the entertainment industry, analyze audience sentiment through voting distributions, or practice advanced feature engineering.

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `title` | Text / String | The name of the movie or TV series (e.g., *The Shawshank Redemption*). |
| `titleType` | Categorical | The format of the content; primarily split into `movie` (82%) and `tvSeries` (13%). |
| `genres` | Text (Multi-label) | A comma-separated list of genres applicable to the title (e.g., `Action, Crime, Drama`). |
| `numVotes` | Numeric (Integer) | The total number of user votes received on IMDb. Highly right-skewed. |
| `year` | Numeric (Integer) | The release year of the title, ranging from 1902 up to 2026. |
| `averageRating`| Numeric (Float) | The weighted average user rating on a scale from 1.0 to 10.0. |

## **Predictive Modeling : Average Rating**
A regression model can be used as a baseline to predict the average rating of a movie/show, using features such as genre, No. of votes and title type. The model's performance and accuracy can be evaluated and advanced tree-based models such as a Random Forest or XGBoost can be developed to further improve accuracy and performance

### **Content-based Recommendation System**
We cna vectorise the `genre` column using a Multi-label Binarizer to combine the genre and title type strings. This can be used to recommend the top 5 closest vectors to the input title by the user. For example, if a user inputs 'Interstellar', the algorithm is able to search through the vector dataset and recommend the top-5 closest titles to interstellar that a user would enjoy

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abbas829/imdb-top-10000-movies-and-tv-series-dataset/title.combined.json
/kaggle/input/datasets/abbas829/imdb-top-10000-movies-and-tv-series-dataset/title.combined.csv


In [2]:
data = pd.read_csv('/kaggle/input/datasets/abbas829/imdb-top-10000-movies-and-tv-series-dataset/title.combined.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          9999 non-null   object 
 1   titleType      9999 non-null   object 
 2   genres         9998 non-null   object 
 3   numVotes       9999 non-null   int64  
 4   year           9999 non-null   int64  
 5   averageRating  9999 non-null   float64
dtypes: float64(1), int64(2), object(3)
memory usage: 468.8+ KB


In [3]:
data.head(10)

,title,titleType,genres,numVotes,year,averageRating
0,The Shawshank Redemption,movie,Drama,3185932,1994,9.3
1,The Dark Knight,movie,"Action,Crime,Drama",3165032,2008,9.1
2,Inception,movie,"Action,Adventure,Sci-Fi",2814573,2010,8.8
3,Game of Thrones,tvSeries,"Action,Adventure,Drama",2611980,2011,9.2
4,Breaking Bad,tvSeries,"Crime,Drama,Thriller",2611328,2008,9.5
5,Fight Club,movie,"Crime,Drama,Thriller",2605573,1999,8.8
6,Interstellar,movie,"Adventure,Drama,Sci-Fi",2525474,2014,8.7
7,Forrest Gump,movie,"Drama,Romance",2494835,1994,8.8
8,Pulp Fiction,movie,"Crime,Drama",2433805,1994,8.8
9,The Matrix,movie,"Action,Sci-Fi",2245953,1999,8.7


In [4]:
data.isnull().sum()

title            0
titleType        0
genres           1
numVotes         0
year             0
averageRating    0
dtype: int64

In [5]:
data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          9999 non-null   object 
 1   titleType      9999 non-null   object 
 2   genres         9998 non-null   object 
 3   numVotes       9999 non-null   int64  
 4   year           9999 non-null   int64  
 5   averageRating  9999 non-null   float64
dtypes: float64(1), int64(2), object(3)
memory usage: 468.8+ KB


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

# preprocessing
df = data.drop(columns=['title', 'averageRating'])

# 2. One-hot encode the categorical variables
df = pd.get_dummies(df, columns=['titleType'], drop_first=True)

# 3. Handle the multi-label 'genres' column by converting it to binary columns
df = df.drop(columns=['genres']).join(data['genres'].str.get_dummies(sep=','))

X = df.values
y = data['averageRating'].values
linreg = LinearRegression()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
linreg.fit(X_train, y_train)

y_pred = linreg.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Model Evaluation Results ===")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R^2 Score (Variance Explained): {r2:.4f}")

=== Model Evaluation Results ===
Root Mean Squared Error (RMSE): 0.7894
R^2 Score (Variance Explained): 0.4128


In [7]:
from sklearn.ensemble import RandomForestRegressor

# Initialize a non-linear tree-based model (Updated)
rf_reg = RandomForestRegressor(
    n_estimators=150, 
    max_depth=10,          # Prevents trees from growing infinitely deep
    min_samples_leaf=5,     # Ensures splits represent a generalized group
    random_state=42, 
    n_jobs=-1
)


rf_reg.fit(X_train, y_train)
y_rf_pred = rf_reg.predict(X_test)

# Evaluate
rf_rmse = root_mean_squared_error(y_test, y_rf_pred)
rf_r2 = r2_score(y_test, y_rf_pred)

print("=== Random Forest Evaluation Results ===")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R^2 Score: {rf_r2:.4f}")

=== Random Forest Evaluation Results ===
RMSE: 0.7693
R^2 Score: 0.4424
